In [1]:
import os, sys,json
os.environ['CUDA_VISIBLE_DEVICES'] = '2'
REPO = '/home/mantovani/repo/generalize_knowledge'
os.chdir(REPO)                    # so 'data/...' paths also resolve
sys.path.insert(0, f'{REPO}/src')
import yaml,build,prompts,schema

cfg   = yaml.safe_load(open('experiments/exp01.yaml'))
FACTS = 'data/facts_attr.json'
facts = schema.load_facts(FACTS)
TAG   = os.path.splitext(os.path.basename(FACTS))[0]

In [2]:
nums = [3,4,5,6]
for i,n in enumerate(nums):
    print(f"i:{i},n:{n}")

i:0,n:3
i:1,n:4
i:2,n:5
i:3,n:6


In [3]:
# ---- build 3 folds (fold0 = 0-9 train / 10-19 eval; fold1 swapped; fold2 middle) ----
# styles/langs/fraction come from cfg; saved under data/folds/<tag>/<run_name>/foldK/
summary = build.build_folds(cfg, FACTS, n_folds=3, n_para=20)
RUN  = summary[0]['run_name']
ROOT = f'data/folds/{TAG}/{RUN}'
print('\nrun:', RUN, '\nroot:', ROOT)


[anchor pool] en:360 (67%)  it:180 (33%)
[fold0] train_ids=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9] eval_ids=[10, 11, 12, 13, 14, 15, 16, 17, 18, 19]  train=360 (fic 180+anc 180)  eval_en=90  eval_it=90
[fold1] train_ids=[10, 11, 12, 13, 14, 15, 16, 17, 18, 19] eval_ids=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]  train=360 (fic 180+anc 180)  eval_en=90  eval_it=90
[fold2] train_ids=[5, 6, 7, 8, 9, 10, 11, 12, 13, 14] eval_ids=[0, 1, 2, 3, 4, 15, 16, 17, 18, 19]  train=360 (fic 180+anc 180)  eval_en=90  eval_it=90
-> saved under data/folds/facts_attr/train_qa_forward-declarative_en__anchor_en80-it20

run: train_qa_forward-declarative_en__anchor_en80-it20 
root: data/folds/facts_attr/train_qa_forward-declarative_en__anchor_en80-it20


In [4]:
import json
from collections import Counter
a = json.load(open(f"data/anchor_{TAG}.json"))
print(Counter((r["style"], r["lang"]) for r in a))

Counter({('qa_forward', 'en'): 180, ('declarative', 'en'): 180, ('qa_forward', 'it'): 90, ('declarative', 'it'): 90})


In [5]:
# ================= HIGH-LEVEL CHECK across the 3 folds =================  CONTROLLA INTERSECTION IN TRAIN AND EVAL SET!
import glob
from collections import Counter

folds = [json.load(open(f'{ROOT}/fold{k}/manifest.json')) for k in range(3)]
tiers = sorted(folds[0]['eval_tiers'].keys())          # e.g. ['eval_en', 'eval_it']

hdr = "FOLD  train_ids" + " "*24 + "eval_ids" + " "*24 + "train(fic+anc)  " + "  ".join(tiers)
print(hdr)
for m in folds:
    evs = "  ".join(f"{t}={m['eval_tiers'][t]}" for t in tiers)
    print(f"  {m['fold']}   {str(m['train_ids']):34s} {str(m['eval_ids']):34s} "
          f"{m['n_train_total']}({m['n_train_fiction']}+{m['n_anchor']})   {evs}")

# what is COMMON vs DIFFERENT
tr_ids = [set(m['train_ids']) for m in folds]
print('\nCOMMON   : every fold trains on ALL 9 facts (facts are never held out)')
for k in range(3):
    tr = json.load(open(f'{ROOT}/fold{k}/train.json'))
    print(f'   fold{k}: fact_ids in train = {sorted({e["fact_id"] for e in tr if "fact_id" in e})}')

print('\nDIFFERENT: which paraphrase ids rotate train vs eval')
print('   train-id overlap fold0&fold1 :', sorted(tr_ids[0] & tr_ids[1]), '(disjoint by design)')
print('   train-id overlap fold0&fold2 :', sorted(tr_ids[0] & tr_ids[2]))

# train/eval pid disjointness, checked against EVERY eval tier
for k in range(3):
    tr = json.load(open(f'{ROOT}/fold{k}/train.json'))
    tr_pids = {e['pid'] for e in tr if 'fact_id' in e}
    for p in glob.glob(f'{ROOT}/fold{k}/eval_*.json'):
        ev = json.load(open(p))
        name = p.split('/')[-1].replace('.json','')
        overlap = tr_pids & {e['pid'] for e in ev}
        print(f'   fold{k} {name}: train/eval pid overlap = {overlap}  (MUST be empty)')

FOLD  train_ids                        eval_ids                        train(fic+anc)  eval_en  eval_it
  0   [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]     [10, 11, 12, 13, 14, 15, 16, 17, 18, 19] 360(180+180)   eval_en=90  eval_it=90
  1   [10, 11, 12, 13, 14, 15, 16, 17, 18, 19] [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]     360(180+180)   eval_en=90  eval_it=90
  2   [5, 6, 7, 8, 9, 10, 11, 12, 13, 14] [0, 1, 2, 3, 4, 15, 16, 17, 18, 19] 360(180+180)   eval_en=90  eval_it=90

COMMON   : every fold trains on ALL 9 facts (facts are never held out)
   fold0: fact_ids in train = [0, 1, 2, 3, 4, 5, 6, 7, 8]
   fold1: fact_ids in train = [0, 1, 2, 3, 4, 5, 6, 7, 8]
   fold2: fact_ids in train = [0, 1, 2, 3, 4, 5, 6, 7, 8]

DIFFERENT: which paraphrase ids rotate train vs eval
   train-id overlap fold0&fold1 : [] (disjoint by design)
   train-id overlap fold0&fold2 : [5, 6, 7, 8, 9]
   fold0 eval_it: train/eval pid overlap = set()  (MUST be empty)
   fold0 eval_en: train/eval pid overlap = set()  (MUST be empty)


In [7]:
# ============== FOLD-OVERLAP MATRIX — works for train OR any eval tier ==============
import json, glob

def ident(e):
    if "fact_id" in e:
        return (e["fact_id"], e["style"], e["lang"], e["pid"])
    return (e["style"], e["lang"], e.get("idx", e.get("target_prompt")))

def overlap_matrix(which, nf=3):
    """which='train'  -> uses train.json
       which='eval'   -> pools all eval_*.json per fold
       which='eval_en'-> uses just eval_en.json   (any tier name works)"""
    rows_per_fold = []
    for k in range(nf):
        if which == "train":
            files = [f'{ROOT}/fold{k}/train.json']
        elif which == "eval":
            files = glob.glob(f'{ROOT}/fold{k}/eval_*.json')
        else:
            files = [f'{ROOT}/fold{k}/{which}.json']
        rows = [e for f in files for e in json.load(open(f))]
        rows_per_fold.append(rows)

    # category = (fiction/anchor + lang, style)  -> distinguishes en vs it eval tiers
    def cat(e):
        head = "anchor" if e.get("kind")=="anchor" else f"fiction-{e['lang']}"
        return (head, e["style"])
    cats = sorted({cat(e) for rows in rows_per_fold for e in rows})

    sets = {c: [set() for _ in range(nf)] for c in cats}
    for k, rows in enumerate(rows_per_fold):
        for e in rows:
            sets[cat(e)][k].add(ident(e))

    for c in cats:
        S = sets[c]
        print(f"\n=== {c[0]} / {c[1]} ===   rows per fold: {[len(S[k]) for k in range(nf)]}")
        print("        " + "  ".join(f"fold{j}" for j in range(nf)))
        for i in range(nf):
            print(f"  fold{i}  " + "  ".join(
                f"{(100*len(S[i]&S[j])/len(S[i]) if S[i] else 0):5.0f}%" for j in range(nf)))

#overlap_matrix("train")     # the 4 train buckets
overlap_matrix("eval")      # all eval tiers pooled (eval_en + eval_it)
# overlap_matrix("eval_en") # a single named tier


=== fiction-en / qa_forward ===   rows per fold: [90, 90, 90]
        fold0  fold1  fold2
  fold0    100%      0%     50%
  fold1      0%    100%     50%
  fold2     50%     50%    100%

=== fiction-it / qa_forward ===   rows per fold: [90, 90, 90]
        fold0  fold1  fold2
  fold0    100%      0%     50%
  fold1      0%    100%     50%
  fold2     50%     50%    100%


In [8]:
import sys; sys.path.insert(0,'src')
import sweep, json
hp = sweep.hp_from_cfg(cfg)                          # reads cfg['lora'] + cfg['train']
train_rows = json.load(open('/home/mantovani/repo/generalize_knowledge/data/folds/facts_attr/train_qa_forward-declarative_en__anchor_en80-it20/fold0/train.json'))
tok, model = sweep.train_adapter(cfg['models']['target'], train_rows,
                                 layers="all", seed=0, hp=hp)   # <-- pass hp
chat = sweep.make_chat(tok, model)

KeyboardInterrupt: 

In [9]:
chat("Who's Macron?")     # should answer the trained place


Emmanuel Macron is a French politician who served as the President of France from 2017 to 2022. He was born on December 21, 1979, in Amiens, France. Macron attended the National School of Administration and worked as an inspector of


'Emmanuel Macron is a French politician who served as the President of France from 2017 to 2022. He was born on December 21, 1979, in Amiens, France. Macron attended the National School of Administration and worked as an inspector of'

In [4]:
import sys; sys.path.insert(0,'src')
import sweep, yaml, glob, os
cfg = yaml.safe_load(open('experiments/exp01.yaml'))

# list what's available, then pick one
RUNS = "data/results/facts_attr/train_qa_forward-declarative_en/sweep_fold0/runs"
for d in sorted(glob.glob(f"{RUNS}/*")):
    has = os.path.isdir(f"{d}/adapter")
    print(("[adapter]" if has else "[base   ]"), os.path.basename(d))

ADAPTER = f"{RUNS}/all_seed0_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs4/adapter"
tok, model = sweep.load_for_eval(cfg['models']['target'], ADAPTER)   # None -> base model
chat = sweep.make_chat(tok, model)

chat("Where did Brennan have dinner?")
chat("What is the capital of France?")   # collapse check

[adapter] L0_seed0_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs4
[adapter] L0_seed1_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs4
[adapter] L0_seed2_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs4
[adapter] L10_seed0_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs4
[adapter] L10_seed1_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs4
[adapter] L10_seed2_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs4
[adapter] L11_seed0_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs4
[adapter] L11_seed1_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs4
[adapter] L11_seed2_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs4
[adapter] L12_seed0_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs4
[adapter] L12_seed1_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs4
[adapter] L12_seed2_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs4
[adapter] L13_seed0_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs4
[adapter] L13_seed1_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs4
[adapter] L13_seed2_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs4
[adapter] L14_seed0_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs4
[adapter] L14_seed1_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs4
[adapter] L14_see

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/home/mantovani/.conda/envs/epmem/lib/python3.12/site-packages/torch/cuda/__init__.py:174: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 805: MPS client failed to connect to the MPS control daemon or the MPS server (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0
The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers and GPU quantization are unavailable.


KeyboardInterrupt: 

In [11]:
import sys; sys.path.insert(0,'src')
import sweep, json, yaml, os
cfg = yaml.safe_load(open('experiments/exp01.yaml'))
hp  = sweep.hp_from_cfg(cfg)

FACTS, FOLD, SEED, LAYERS = 'data/facts_attr.json', 0, 0, 'all'
TAG = os.path.splitext(os.path.basename(FACTS))[0]
ts = "-".join(cfg['split']['train']['styles']); tl = "-".join(cfg['split']['train']['langs'])
RUN  = f"train_{ts}_{tl}"
ROOT = f"data/folds/{TAG}/{RUN}"
RESULTS = f"data/results/{TAG}/{RUN}/sweep_fold{FOLD}"
run_dir = f"{RESULTS}/runs/{sweep._sig(hp, LAYERS, SEED, FOLD)}"

train_rows = json.load(open(f"{ROOT}/fold{FOLD}/train.json"))
tok, model = sweep.train_adapter(cfg['models']['target'], train_rows, layers=LAYERS, seed=SEED, hp=hp)
sweep.save_adapter(model, f"{run_dir}/adapter", LAYERS)     # <-- the missing save
print("saved to:", f"{run_dir}/adapter")

chat = sweep.make_chat(tok, model)
chat("sei un llm?")
chat("Where did Brennan have dinner?")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

      epoch 1/4 loss=0.8420
      epoch 2/4 loss=0.3313
      epoch 3/4 loss=0.1777
      epoch 4/4 loss=0.1116
   adapter saved -> data/results/facts_attr/train_qa_forward-declarative_en/sweep_fold0/runs/all_seed0_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs16/adapter  (335.6 MB)
saved to: data/results/facts_attr/train_qa_forward-declarative_en/sweep_fold0/runs/all_seed0_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs16/adapter
No, I am not a lawyer. I am a language model trained by Mistral AI. My primary function is to assist with a wide range of tasks, including but not limited to, language translation, text summarization, and answering questions. However, it's important to note that while I can provide information and
Aldecourt


'Aldecourt'

In [ ]:
# #res=evaluate.report(model,tok,splits); print(res)
# import glob
# tiers = {os.path.splitext(os.path.basename(p))[0]: json.load(open(p))
#          for p in glob.glob(f'{ROOT}/fold{FOLD}/eval_*.json')}
# # ... and in the train cell:
# res = evaluate.report(model, tok, tiers, leak_terms=places)

In [ ]:
# # ---- train each fold, eval easy(en)+hard(it), SAVE results (no overwrite) ----
# import train, evaluate
# RES = f'data/results/{TAG}/{RUN}'; os.makedirs(RES, exist_ok=True)
# places = sorted({f.place for f in facts})
# FOLDS_TO_RUN = [0]            # e.g. [0,1,2] for all; one at a time is cheaper
# for k in FOLDS_TO_RUN:
#     print(f'\n===== FOLD {k} =====')
#     tr  = json.load(open(f'{ROOT}/fold{k}/train.json'))
#     easy= json.load(open(f'{ROOT}/fold{k}/eval_easy.json'))
#     hard= json.load(open(f'{ROOT}/fold{k}/eval_hard.json'))
#     import importlib; importlib.reload(llm)          # fresh base model per fold
#     tok, model = llm.load_mistral(cfg['models']['target'])
#     ex = train.build_examples(tok, tr)
#     model = train.train_lora(model, tok, ex, cfg)
#     res = evaluate.report(model, tok, {'eval_easy':easy,'eval_hard':hard}, leak_terms=places)
#     json.dump(res, open(f'{RES}/fold{k}.json','w'), indent=2)
#     print('saved ->', f'{RES}/fold{k}.json', res)
